In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
#Define order configuration
SEED = 20261022
ORDER_COUNT = 4000

print("Seed:", SEED)
print("Total Orders:", ORDER_COUNT)

########Generate order records

In [0]:

orders = spark.range(1, ORDER_COUNT + 1)

orders = orders.withColumn(
    "order_id",
    F.concat(
        F.lit("ORD-2026-"),
        F.lpad(F.col("id").cast("string"), 6, "0")
    )
)



In [0]:
# Select a customer using deterministic hashing

CUSTOMER_COUNT = 1000

orders = orders.withColumn(
    "customer_number",
    F.pmod(
        F.xxhash64("order_id", F.lit(SEED)),
        F.lit(CUSTOMER_COUNT)
    ) + 1
)

orders = orders.withColumn(
    "customer_id",
    F.concat(
        F.lit("CUS-"),
        F.lpad(F.col("customer_number").cast("string"), 6, "0")
    )
)



In [0]:
# Generate order dates

orders = orders.withColumn(
    "order_date",
    F.date_add(
        F.lit("2025-01-01").cast("date"),
        F.pmod(
            F.xxhash64("order_id", F.lit("date")),
            F.lit(668)
        ).cast("int")
    )
)



In [0]:
# Generate order status

orders = orders.withColumn(
    "order_status",
    F.when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 85, "DELIVERED")
     .when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 93, "RETURNED")
     .when(F.pmod(F.xxhash64("order_id", F.lit("status")), F.lit(100)) < 97, "CANCELLED")
     .otherwise("PENDING")
)



In [0]:
# Generate payment methods

orders = orders.withColumn(
    "payment_method",
    F.when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 0, "UPI")
     .when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 1, "CARD")
     .when(F.pmod(F.xxhash64("order_id", F.lit("payment")), F.lit(4)) == 2, "NET_BANKING")
     .otherwise("COD")
)

order_df = orders.select(
    "order_id",
    "customer_id",
    "order_date",
    "order_status",
    "payment_method"
)

display(order_df.limit(10))

In [0]:
#Validate the order data
print("Total Orders:", order_df.count())

In [0]:
#Check duplicate order IDs:
order_df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
#Check customer foreign keys:
invalid_customers = order_df.join(
    spark.read.table("customer_master").select("customer_id"),
    on="customer_id",
    how="left_anti"
)

print("Invalid Customer References:", invalid_customers.count())

In [0]:
#Check order statuses:
order_df.groupBy("order_status").count().show()

In [0]:
#Save the Order table
order_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("order_master")

In [0]:
%sql
SELECT * FROM order_master LIMIT 10;